# 📓 Semana 3 · Dia 3 — Transformações: select, withColumn, joins e agregações

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (ELT with Spark) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Pipeline de transformação limpo rodando |

---


## 📖 Teoria — As transformações do dia a dia

| Operação | Uso |
|---|---|
| `select` | escolher colunas |
| `withColumn` | criar/alterar coluna (com expressão) |
| `filter` / `where` | filtrar linhas |
| `groupBy().agg()` | agregações |
| `join` | combinar tabelas |
| `union` / `unionByName` | empilhar tabelas |
| `dropDuplicates` | deduplicar |
| `orderBy` | ordenar |

**Regra de ouro**: prefira expressões de coluna (Spark SQL) a UDFs Python — expressões são otimizadas pelo Catalyst e executadas em JVM; UDFs Python são centenas de vezes mais lentas.


## 📖 Teoria — Tipos de join

| Join | Devolve | Uso típico |
|---|---|---|
| inner | só correspondências | padrão |
| left | tudo da esquerda + match à direita | enriquecer |
| right | tudo da direita + match à esquerda | raro |
| full | tudo de ambos | auditoria |
| left semi | linhas da esquerda com match | filtro eficiente |
| left anti | linhas da esquerda SEM match | exclusão/validação |

> 🎯 **Dica de prova**: `left semi` = filtro (sem colunas da direita); `left anti` = exclusão. São perguntas garantidas na DEA.


### 💻 Na prática — Pipeline de transformação

Construa um pipeline ELT sobre o Bronze: limpeza, enriquecimento e agregação.


In [ ]:
# Ler o Bronze
from pyspark.sql.functions import col, when, upper, round as r, sum as s, to_date
df = spark.table("workspace.bronze.vendas_bronze")
print("Linhas:", df.count())

In [ ]:
# Limpeza e enriquecimento
df_enriquecido = (df
    .filter(col("CustomerID").isNotNull())
    .withColumn("pais_upper", upper(col("Country")))
    .withColumn("receita_linha", r(col("Quantity") * col("UnitPrice"), 2))
    .withColumn("categoria_preco",
        when(col("UnitPrice") < 2, "barato")
        .when(col("UnitPrice") < 20, "medio")
        .otherwise("caro")))
df_enriquecido.select("InvoiceNo", "Country", "pais_upper", "receita_linha", "categoria_preco").show(5, truncate=False)

In [ ]:
# Agregação + join
receita_por_pais = (df_enriquecido
    .groupBy("pais_upper")
    .agg(s("receita_linha").alias("receita_total"))
    .orderBy(col("receita_total").desc()))
receita_por_pais.show(5)

### 💻 Na prática — Joins na prática

Crie uma dimensão pequena de categoria e faça os joins.


In [ ]:
# Dimensão de exemplo (categoria por StockCode)
categorias = spark.createDataFrame([
    ("85123A", "Decoração"), ("71053", "Cozinha"), ("84406B", "Papelaria"),
    ("22423", "Iluminação"), ("47566", "Vestuário"),
], ["StockCode", "categoria"])
# inner: só produtos com categoria conhecida
df_inner = df_enriquecido.join(categorias, "StockCode", "inner")
print("inner:", df_inner.count())
# left anti: produtos sem categoria cadastrada
df_sem_cat = df_enriquecido.join(categorias, "StockCode", "left_anti")
print("left_anti (sem categoria):", df_sem_cat.select("StockCode").distinct().count())

> 🎯 **Dica de prova**: UDF Python vs expressão: a prova pergunta por que expressões são preferíveis (performance; Catalyst otimiza; execução JVM). Evite UDFs quando houver função nativa.


## 🎯 Exercícios de fixação

**1.** Use `dropDuplicates(['CustomerID'])` e explique o que acontece.

**2.** Faça um join left e diga quantas linhas ficam com categoria nula.

**3.** Crie coluna `faixa` com when: receita < 50 → 'baixa', < 200 → 'media', senão 'alta'.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** dropDuplicates

Remove linhas duplicadas **na combinação das colunas indicadas** (aqui, um registro por cliente, mantendo o primeiro). Para dedup por chave é padrão em camada Prata.

**2.** Left join com nulos

`df.join(cat, 'StockCode', 'left')` — linhas da esquerda sem match ficam com `categoria` nula. Conte com `filter(col('categoria').isNull()).count()`.

**3.** Faixa

```python
.withColumn('faixa', when(col('receita_linha') < 50, 'baixa').when(col('receita_linha') < 200, 'media').otherwise('alta'))
```



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*